# Download data voor laag "ruw" en maak er parquet files van voor laag "brons"

Auteurs: Niels Molenaar, Maurice de Kleijn, Thya van den Berg

Organisatie: Rijksdienst voor het cultureel erfgoed

Programma: DNA-NL

Project: Bronnen- en datakwaliteit

Projectleider: Maurice de Kleijn

De data die wordt gedownload bestaat voornamelijk uit data van de RCE en deels uit openbare data van andere nederlandse overheidsinstanties. In de "ruw"-laag, worden de bestanden neergezet precies zoals ze bestaan op de plek waar ze vandaan worden gehaald (bijvoorbeeld het Nationaal Georegister, PDOK, of de RCE G: schijf). In de "brons"-laag wordt dezelfde data opgeslagen, maar dan allemaal in hetzelfde bestandsformaat: parquet. Dit wordt gedaan zodat in de andere notebooks met alle bestanden hetzelfde kan worden omgegaan, in dit geval met SQL queries in duckdb.


Parquet is een bestandsformaat dat informatie opslaat in de vorm van tabellen. Anders dan andere file formats die dit doen, worden parquet files per kolom opgeslagen, en niet per rij. Het is een open-source bestandsformaat gecreeerd door Apache.

zie: https://parquet.apache.org/

zie: https://en.wikipedia.org/wiki/Apache_Parquet

# Importeer packages

In [283]:
# # installeer packages als nodig. Dit hoeft maar eenmaal per opzet van de environment
# %pip install unidecode
# %pip install feedparser
# %pip install geopandas
# %pip install pyarrow
# %pip install requests
# %pip install pyodbc
# %pip install sqlalchemy
# %pip install sqlalchemy_access
# # soms is er een probleem met het installeren/importeren van sqlalchemy_acces, zie https://stackoverflow.com/questions/18907889/importerror-no-module-named-pywintypes voor de oplossing
# # als je een .venv gebruikt, zet dan de twee genoemde .dll files ook in de site_packages van je .venv
# # dependencies sqlalchemy_access
# %pip install pywintypes
# %pip install owslib

In [284]:
# importeer packages
from datetime import datetime
from pathlib import Path
import shutil
import re
from unidecode import unidecode
from string import Template
import os
import datetime
import pandas as  pd
import geopandas as gpd
import zipfile
import feedparser
import requests
import xml.etree.ElementTree as ET
import pyodbc
import sqlalchemy
import sqlalchemy_access
print([x for x in pyodbc.drivers()]) # als de Microsoft Access Drivers er niet zijn is het laden van de access db niet mogelijk: https://github.com/mkleehammer/pyodbc/wiki/Connecting-to-Microsoft-Access
from owslib.wfs import WebFeatureService
from owslib.fes import FilterRequest

['SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)']


## Maak functies
Functies zijn stukjes code die verderop in het document kunnen worden hergebruikt. Code definiëren in functies zorgt ervoor dat de rest van je code minder lang hoeft te zijn, waardoor het makkelijker te lezen is.

### functies die te maken hebben met naamgeving

In [285]:
def toDatabaseNamingConventions(string):
    """Normaliseert tabel en kolomnamen"""
    string = removeDiacritics(string)
    string = replaceInvalidCharacters(string)
    string = toSnakeCase(string)
    return string

#Stap2 - DatabaseNamingConventions
def replaceInvalidCharacters(string):
    """Vervangt speciale karakters met underscore"""
    disallowed_characters = r"!@#$%^&*()[]{};:,./<>?\|`~-=_+"
    for character in disallowed_characters:
        string = string.replace(character, "_")
    return string

#Stap3 - DatabaseNamingConventions
def toSnakeCase(string):
    """Vervangt alles behalve letters en cijfers met underscore, dubbele underscores worden enkel"""
    string = re.sub(r'(?<=[a-z])(?=[A-Z])|[^a-zA-Z0-9]', ' ', string).strip().replace(' ', '_').replace('__', '_')
    return ''.join(string.lower())

#Stap1 - DatabaseNamingConventions
def removeDiacritics(string):
    """Vervangt letter met accenten"""
    return unidecode(string)


### functies die te maken hebben met het ophalen van de tijd/datum van laatste wijziging

In [286]:

def modification_date(file_path):
    """Leest de datum van laatste modificatie uit de metadata van een bestand"""
    t = os.path.getmtime(file_path)
    print(t)
    return datetime.datetime.fromtimestamp(t)



# verkrijg de timestamp uit een atom feed behoende het opgegeven databsetand of archiveringsbestand
def get_timestamp_from_atom_feed(databron_archiveerbestand_atom_feed, databron_databestand_atom_feed, databron_archiveerbestand_naam, databron_databestand_naam):
    """Verkrijgt de datum van laatste modificatie voor een bestand afkomstig uit een ATOM feed (zoals PDOK)"""
    if databron_archiveerbestand_atom_feed != "":
        atom_url = databron_archiveerbestand_atom_feed
    elif databron_databestand_atom_feed != "":
        atom_url = databron_databestand_atom_feed
    else:
        pass

    if databron_archiveerbestand_naam != "":
        filename = databron_archiveerbestand_naam
    elif databron_databestand_naam != "":
        filename = databron_databestand_naam
    else:
        pass  

    timestamp_str = ""

    # De feed parsen
    feed = feedparser.parse(atom_url)
    
    # Controleren op fouten
    if feed.bozo == 0:  
        # Door de entries (datasets) lopen
        for entry in feed.entries:
            print(entry.links)
            for link in entry.links:
                dataset_bestandsnaam = link['href'].split('/')[-1]
                print(filename, dataset_bestandsnaam)
                if dataset_bestandsnaam == filename:
                    timestamp_str = entry.updated
                    print(timestamp_str)
        timestamp_object = datetime.datetime.fromisoformat(timestamp_str)
        return timestamp_object   
    else:
        print("Fout bij het parsen van de feed:", feed.bozo_exception)


def get_timestamp_from_ngr_rest_api(metadata_code):
    """Haalt uit de metadata API van het nationaal georegister de modification date, of als deze niet beschikbaar is, de maintainance frequency"""

    # TODO does NOT work. Somehow the API keeps returning html instead of xml. temporarily return today's date
    if metadata_code == "":
        raise ValueError("metadata_code is empty")
    api_url = "https://www.nationaalgeoregister.nl/geonetwork/srv/api/records/"
    record_view_formatter = "?recordViewFormatter=application%2Fxml"
    api_metadata_record = api_url + metadata_code + record_view_formatter
    xml = requests.get(api_metadata_record)
    xml.raise_for_status()
    root = ET.fromstring(xml.content)
    return ET.tostring(root, encoding = 'utf-8', method = 'xml')
    name_space = {
    "gmd": "http://www.isotc211.org/2005/gmd",
    "gco": "http://www.isotc211.org/2005/gco",
    }
    # TODO toevoegen dat ook date maintained kan worden opgehaald
    # date_maintained = root.find(".//gmd:")

    update_frequency = root.find(
    ".//gmd:maintenanceAndUpdateFrequency/"
    "gmd:MD_MaintenanceFrequencyCode",
    name_space,
    )
    return update_frequency, api_metadata_record

def get_most_recent_mutation_wfs(url, mutation_date_colname, databron_wfs_parameters):
    """Haalt de meest recente mutatiedatum van een feature uit een WFS service op"""

    pass

### Functies die te maken hebben met het downloaden, verplaatsen en verder behandelen van bestanden

In [287]:
def copy_file(source, target, type):
    """Kopieert een bestand van de bron naar een beoogde map"""
    # get the directory out of the path
    dir_name, _ = os.path.split(target)
    # create directory if does not exist
    os.makedirs(dir_name,exist_ok=True)
    # copy file form sopurce to target
    print(source, target)
    if type == "ESRI File Geodatabases":
        # ESRI File Geodatabases is geen bestand maar een folder met bestanden
        shutil.copytree(source, target)
    elif type == "Shapefile":
        # ESRI shapefile bestaat uit meerdere bestanden, maar niet altijd dezelfde
        list_of_shapefile_filetypes = [".shp", ".cpg", ".dbf", ".prj", ".sbn", ".sbx", ".shx"]
        for item in list_of_shapefile_filetypes:
            source_file = source[:-4]
            source_file = source_file+item
            print(source_file)
            target_file = target[:-4]
            target_file = target_file+item
            print(target_file)
            shutil.copy(source_file, target_file)

    else:
        print("arrived at else")
        shutil.copy(source, target)

def read_pw_from_file(file_path):
    """Leest een password uit een tekstbestand"""
    with open(file_path, 'r') as file:
        pw = file.read()
    if isinstance(pw, str):
        return pw
    else:
        raise ValueError("file did not contain a string")

def zip_has_pwd(path):
    """Checkt of een zip bestand is beveiligd met een password"""
    zf = zipfile.ZipFile(path)
    for zinfo in zf.infolist():
        is_encrypted = zinfo.flag_bits & 0x1
        if is_encrypted:
            #print('%s is encrypted!' % zinfo.filename)
            return True
        else:
            return False

def remove_pwd_from_zip(source_zip: os.PathLike, source_pwd, skip_existing=True):
    """Verwijderd password van zip"""
    # geeft de bestand naam een underscore als perfix
    new_zip = f"{os.path.split(source_zip)[0]}/_{os.path.split(source_zip)[1]}"
    with zipfile.ZipFile(source_zip, 'r') as source_archive:
        with zipfile.ZipFile(new_zip, 'a') as target_archive:
            for file in source_archive.infolist():
                print(file)
                if skip_existing and zipfile.Path(target_archive, file.filename).exists():
                    print(f"File '{file.filename}' already exists in target archive. Skipping...")
                else:
                    target_archive.writestr(file, source_archive.open(file, pwd = bytes(source_pwd,'utf-8')).read())
        return new_zip

def gis_file_to_parquet(db_zipfile_path, db_file_path, whitelist):
    """Converteert een (GIS) database bestand naar parquet.
    Op dit moment geschikt voor (al dan niet eerst gezipt):
    .gpkg, .gml, .shp, .gdb, .geoJSON
    """

    # get the directory out of the path
    if db_zipfile_path != "":
        # het is een zip bestand

            # het is een GIS bestand
        geofile_dir_name, _ = os.path.split(db_zipfile_path)
        geofile_path = f"zip://{db_zipfile_path.replace('\\', '/')}" # gdp vereist forward slashes
        layers_list = gpd.list_layers(geofile_path)['name']
    elif db_file_path != "":
        # het is geen zip bestand
        geofile_dir_name, _ = os.path.split(db_file_path)
        geofile_path = f"{db_file_path}"
        layers_list = gpd.list_layers(geofile_path)['name']
    else:
        pass
    if whitelist == {}:
        # zonder whitelist worden van shapefile alle lagen en entiteiten verwerkt
        for layer in layers_list:
            print("layer", layer)
            layer_df = gpd.read_file(geofile_path, layer=layer)
            #print(layer_df.columns.tolist())
            columns = layer_df.columns
            #print(columns)
            # entiteit namen normaliseren naar database naam conventie
            for column in columns:
                old_column_name = str(column)
                new_column_name = toDatabaseNamingConventions(str(column))
                layer_df = layer_df.rename(columns = {old_column_name:new_column_name})
            columns = layer_df.columns
            #print(columns)
            #print(columns)
            #layer_df.columns = layer_df.columns.str.lower()
            #print(layer_df.columns.tolist())
            #layer_df.to_file(f"./{jaar}/{maand}/{dag}/{layer}.parquet")
            layer = toDatabaseNamingConventions(layer)
            #print(f"{geofile_dir_name}/{layer}.parquet")
            # maak mappen aan als deze nog niet bestaan
            #print(geofile_dir_name)
            Path(geofile_dir_name).mkdir(parents=True, exist_ok=True)
            #laag opslaan in een parquet file
            datalake_parquet_path = os.path.normpath(f"{datalake_path}/{bronzen_laag_path}/{database_naam}/{layer}.parquet")
            Path(f"{datalake_path}/{bronzen_laag_path}/{database_naam}").mkdir(parents=True, exist_ok=True)
            layer_df.to_parquet(datalake_parquet_path)
    else:
        # van de shapefile worden alleen de lagen en entiteiten uit de whitelist verwerkt
        layers_whitelist = list(whitelist.keys())
        # data kwaliteit: lijst van lagen die ontbreken in de shapefile
        layers_missinglist = list(set(layers_whitelist) - set(layers_list))
        if len(layers_missinglist) > 0:
            for layers_missing in layers_missinglist:
                print(f"ERROR: laag '{layers_missing}' ontbreekt in databestand {database_naam}.")
        # als alle lagen uit de whitelist aanwezig zijn
        else:
            for layer in layers_whitelist:
                #print("layer", layers_whitelist)
                layer_df = gpd.read_file(geofile_path, layer=layer)
                columns = layer_df.columns
                columns_list = layer_df.columns.tolist()
                columns_whitelist = whitelist[layer]
                columns_blacklist = list(set(columns_list) - set(columns_whitelist))

                # data kwaliteit: lijst van kolommen die ontbreken
                columns_missinglist = list(set(columns_whitelist) - set(columns_list))
                if len(columns_missinglist) > 0:
                    for column_missing in columns_missinglist:
                        print(f"ERROR: entiteit '{column_missing}' ontbreekt in tabel {layer} in databestand {database_naam}.")
                #verwijder entiteiten die niet in de whitelist staan
                for column_blacklist in columns_blacklist:
                    layer_df.drop(column_blacklist, axis=1, inplace=True)
                #entiteit namen normaliseren naar database naam conventie
                columns = layer_df.columns
                for column in columns:
                    old_column_name = str(column)
                    new_column_name = toDatabaseNamingConventions(str(column))
                    layer_df = layer_df.rename(columns = {old_column_name:new_column_name})
                #laagnaam normaliseren naar database naam conventie
                layer = toDatabaseNamingConventions(layer)
                #laag opslaan in een parquet file
                datalake_parquet_path = os.path.normpath(f"{datalake_path}/{bronzen_laag_path}/{database_naam}/{layer}.parquet")
                print(datalake_parquet_path, 0)
                Path(f"{datalake_path}/{bronzen_laag_path}/{database_naam}").mkdir(parents=True, exist_ok=True)
                layer_df.to_parquet(datalake_parquet_path)

def access_to_parquet(db_zipfile_path, db_file_path, whitelist):
    """
    Converteert een access database (al dan niet gezipt) naar een parquet bestand.
    Werkt waarschijnlijk alleen op Windows vanwege afhankelijkheid drivers voor Microsoft Access

    Let op, versie voor niet gezipt bestand nog niet getest! zitten ongetwijfeld fouten in.

    :param db_zipfile_path str:  path naar het gezipte bestand in de ruw laag.
    :param db_file_path str:  kan óf het path naar het ongezipte bestand zijn, óf de bestandsnaam in de gezipte folder
    :param whitelist list: lijst met te behouden tabellen.

    :return table_names: list: lijst met tabellen die omgezet zijn naar een parquet bestand


    """
    # maak een tijdelijke zip extractie
    if db_zipfile_path != "":

        temp_dir_path = os.path.join(".\\", "temp_zip")

        if not os.path.exists(temp_dir_path):
            # maak directory als deze nog niet bestaat
            os.mkdir(temp_dir_path)

            # Extract naar temp directory
        with zipfile.ZipFile(db_zipfile_path, 'r') as zip_ref:
            zip_ref.extract(os.path.basename(db_file_path), temp_dir_path)

        db_path = os.path.join(temp_dir_path, os.path.basename(db_file_path))
            # maak connectie met de access db
        conn_str = (
                r'DRIVER={Microsoft Access Driver (*.mdb, *.accdb)};'
                r'DBQ=' + db_path + ';'
            )
        conn = pyodbc.connect(conn_str)
        cursor = conn.cursor()

        conn_str = (
            r"DRIVER={Microsoft Access Driver (*.mdb, *.accdb)};"
            f"DBQ={db_path};"
        )
        connection_url = sqlalchemy.engine.URL.create(
            "access+pyodbc",
            query={"odbc_connect": conn_str},
        )
        engine = sqlalchemy.create_engine(connection_url)
        conn = engine.connect()

        if whitelist == {}:
            # maak een lijst van de beschikbare tabellen
            tables = cursor.tables()
            table_names = [table.table_name for table in tables if table.table_type == 'TABLE']
        else:
            table_names = whitelist.keys()
        # maak directory aan
        Path(f"{datalake_path}/{bronzen_laag_path}/{database_naam}").mkdir(parents=True, exist_ok=True)

        # zet elke tabel om in een pandas db en sla deze op als parquet (relaties worden hierbij dus niet behouden)
        for table in table_names:
            if whitelist == {}:
                query = f"SELECT {whitelist[table]} FROM {table};"
            else:
                query = f"SELECT * FROM {table};"
            df = pd.read_sql_query(query, conn)
            datalake_parquet_path = os.path.normpath(f"{datalake_path}/{bronzen_laag_path}/{database_naam}/{table}.parquet")
            df.to_parquet(datalake_parquet_path)
    else:
        # niet gezipt # TODO nog niet getest!
        conn_str = (
                r'DRIVER={Microsoft Access Driver (*.mdb, *.accdb)};'
                r'DBQ=' + db_file_path + ';'
            )
        conn = pyodbc.connect(conn_str)
        cursor = conn.cursor()

        conn_str = (
            r"DRIVER={Microsoft Access Driver (*.mdb, *.accdb)};"
            f"DBQ={db_file_path};"
        )
        connection_url = sqlalchemy.engine.URL.create(
            "access+pyodbc",
            query={"odbc_connect": conn_str},
        )
        engine = sqlalchemy.create_engine(connection_url)
        conn = engine.connect()

        if whitelist == {}:
            # maak een lijst van de beschikbare tabellen
            tables = cursor.tables()
            table_names = [table.table_name for table in tables if table.table_type == 'TABLE']
        else:
            table_names = whitelist.keys()
        # maak directory aan
        Path(f"{datalake_path}/{bronzen_laag_path}/{database_naam}").mkdir(parents=True, exist_ok=True)

        # zet elke tabel om in een pandas db en sla deze op als parquet (relaties worden hierbij dus niet behouden)
        for table in table_names:
            if whitelist == {}:
                query = f"SELECT {whitelist[table]} FROM {table};"
            else:
                query = f"SELECT * FROM {table};"
            df = pd.read_sql_query(query, conn)
            datalake_parquet_path = os.path.normpath(f"{datalake_path}/{bronzen_laag_path}/{database_naam}/{table}.parquet")
            df.to_parquet(datalake_parquet_path)

    # sluit connectie
    conn.commit()
    cursor.close()
    conn.close()
    # verwijder het temp bestand
    os.remove(db_path)
    os.rmdir(temp_dir_path)

    return table_names



## Variabelen definiëren voor Medallion Architecture
Medallion architectuur is een data design patroon dat bedoeld is voor systemen die meerdere bronnen inladen en deze daarna willen combineren. Het gebruikelijke design heeft 3 lagen, hier worden er 4 gedefinieerd:
- ruw (bestanden precies zoals ze binnenkomen)
- brons (de tabellen en databases uit de ruwe bestanden, omgezet naar allemaal hetzelfde bestandsformaat met een opgeschoonde bestandsnaam)
- zilver (dezelfde tabellen en databases, maar opgeschoond op basis van de datakwaliteitsrichtlijnen)
- goud (dezelfde informatie, maar geaggregeerd en gecombineerd op een manier die nuttig is voor de eindgebruiker)

Op basis van waar de file vandaan komt, moet verschillende informatie worden aangeleverd.

In [288]:
datalake_path = os.path.normpath("./data/")
ruwe_laag_path = "ruw"
bronzen_laag_path = "brons"
zilveren_laag_path = "zilver"
gouden_laag_path = "goud"

# uncomment de informatie die je wilt inladen, 1 tegelijkertijd.
# dataset = "Beschermde stads- en dorpsgezichten"    # toegang tot de g-schijf vereist (of downloaden)
# dataset = "Rijksmonumentale boerderijen"           # toegang tot de g-schijf vereist
# dataset = "Rijksmonumentale sluizen en stuwen"     # toegang tot de g-schijf vereist
# dataset = "Rijksbeschermde groenaanleggen"         # toegang tot de g-schijf vereist
# dataset = "Bestuurlijke Gebieden 2026"
# dataset = "Bestuurlijke Gebieden 2025"
# dataset = "Bestuurlijke Gebieden 2024"
# dataset = "Bestuurlijke Gebieden 2023"
# dataset = "Bestuurlijke Gebieden 2022"
# dataset = "Bestuurlijke Gebieden 2021"
# dataset = "Bestuurlijke grenzen 2021"              # !!under construction!!
# dataset = "Bestuurlijke grenzen 2020"              # !!under construction!!
# dataset = "Bestuurlijke grenzen 2019"              # !!under construction!!
# dataset = "Bestuurlijke grenzen 2018"              # !!under construction!!
# dataset = "Bestuurlijke grenzen 2017"              # !!under construction!!
# dataset = "Basisregistratie Adressen en Gebouwen"  # !!kan niet in de blauwe omgeving (vdi) gedraaid worden!!
# dataset = "Publiekrechtelijke Beperkingen"
# dataset = "Rijksmonumentenregister"
# dataset = "rijksmonumentencontouren"
# dataset = "rijksmonumentenpunten"
dataset = "archeologische complexen"


In [289]:
datalake_path

'data'

## Variabelen definiëren voor in te laden bestanden

Dit is een template met alle variabelen die kunnen worden ingevuld. Veel hiervan blijven leeg. De variabelen moeten echter wel worden aangemaakt.

In [290]:
# template code, niet runnen!
#
# if dataset == "": # naam die overeenkomt met 1 uit de lijst met namen uit vorige code chuck
#     dataset_naam = "" # naam waaronder de dataset bekend staat (doorgaans hetzelfde als dataset)
#     database_naam = toDatabaseNamingConventions(dataset_naam) # schoon de naam op
#
#     # de "archiveerbestand" variabelen hoeven alleen te worden gedefinieerd als het bestand binnenkomt in gecomprimeerde vorm, zoals een zipbestand.
#     databron_archiveerbestand_naam = "" # naam van het gecomprimeerde bestand (zonder path)
#     databron_archiveerbestand_type =  "" # type archiveerbestand: zip, tar, etc.
#     databron_archiveerbestand_wachtwoord = read_pw_from_file(os.path.normpath(r"")) #
#     databron_archiveerbestand_api = ""    # fileshare (lokaal/netwerkschijf bestand) of url (download)
#     databron_archiveerbestand_fileshare = os.path.normpath(r"") # path naar het zipbestand, volledig, of vanaf project root, als het om een lokaal/netwerkschijf bestand gaat. ZONDER bestandsnaam!
#     databron_archiveerbestand_url = "" # url naar het zipbestand, als het om een te downloaden bestand gaat
#     databron_archiveerbestand_atom_feed = "" # url van de atom feed, als het bestand van een atom feed wordt gedownload
#
# # De whitelist bevat een dictionary met daarin alle tabellen, en alle kolommen binnen die tabellen, die moeten worden gedownload. Bij GIS bestanden is er altijd een kolom "geometry"
#     whitelist = {
#         'tabel': ['kolom 1', 'kolom2', 'geometry']
#     }
#     # de "databestand" variabelen moeten worden ingevuld voor het in te laden bestand. Als het gaat om een gearchiveerd bestand, gaat het om het bestand binnen de gecomprimeerde map dat de data ook werkelijk bevat.
#     databron_databestand_naam = "" # file naam, inclusief uitgang, zonder path
#     databron_databestand_namen = [] # als er meerdere bestanden gecombineerd gedownload/geontzipt worden, geef hier de lijst van bestanden.
#     databron_databestand_type = "" #type file, kan nu zijn: GeoPackage, Shapefile, ESRI File Geodatabases, GeoJSON, GML of MS access database
#     databron_databestand_wachtwoord = read_pw_from_file(os.path.normpath(r"")) # als het bestand vergrendeld is met een wachtwoord, laad dit hier in vanuit een tekstbestand, of voer handmatig in.
#     databron_databestand_api = ""   # fileshare of url
#     databron_databestand_fileshare = "" # zie hierboven, maar dan voor bestand. ZONDER bestandsnaam!
#     databron_databestand_url = "" # zie hierboven, maar dan voor bestand
#     databron_databestand_atom_feed = "" # zie hierboven, maar dan voor bestand
#     databron_wfs_parameters = "" # als een bestand vanuit een WFS service wordt gedownload, geef hier de parameters.

eerst alle variabelen aanmaken en vullen met lege strings...

In [291]:
dataset_naam = ""
database_naam = toDatabaseNamingConventions(dataset_naam)
databron_archiveerbestand_naam = ""
databron_archiveerbestand_type =  ""
databron_archiveerbestand_wachtwoord = ""
databron_archiveerbestand_api = ""
databron_archiveerbestand_fileshare = ""
databron_archiveerbestand_url = ""
databron_archiveerbestand_atom_feed = ""

whitelist = {}

databron_databestand_naam = ""
databron_databestand_namen = []
databron_databestand_type = ""
databron_databestand_wachtwoord = ""
databron_databestand_api = ""
databron_databestand_fileshare = ""
databron_databestand_url = ""
databron_databestand_atom_feed = ""
databron_wfs_parameters = ""
databron_wfs_metadata_url = ""

In [292]:
# worden bij timestamp gevuld
databron_archiveerbestand_path = ""
databron_databestand_path = ""

### stads-en dorpsgezichten
lokaal/ netwerkschijf gecomprimeerd bestand met wachtwoord

In [293]:
if dataset == "Beschermde stads- en dorpsgezichten":
    dataset_naam = "Beschermde stads- en dorpsgezichten"
    database_naam = toDatabaseNamingConventions(dataset_naam)

    databron_archiveerbestand_naam = "Townscapes.zip"
    databron_archiveerbestand_type =  "zip"
    databron_archiveerbestand_wachtwoord = read_pw_from_file(os.path.normpath(r"data/townscapes_pw.txt"))
    databron_archiveerbestand_api = "fileshare"
    databron_archiveerbestand_fileshare = os.path.normpath(r"data//")

    whitelist = {
        'Townscapes': ['BRON_ID', 'NAAM', 'IN_PROCEDU', 'AANGEWEZEN', 'JURSTATUS', 'DATUM_BEGR', 'ONDERGROND', 'OPPERV_HA', 'OPPERV_KM2', 'CENTROID_X', 'CENTROID_Y', 'IN_PROC_S', 'AANGEWEZ_S', 'BEGRENSD_S', 'KICHLINK', 'Type', 'geometry']
    }

    databron_databestand_naam = "Townscapes.shp"
    databron_databestand_type = "shapefile"


### Rijksmonumentale boerderijen
lokaal bestand/ netwerkschijf

In [294]:
if dataset == "Rijksmonumentale boerderijen":
    dataset_naam = "Rijksmonumentale boerderijen"
    database_naam = toDatabaseNamingConventions(dataset_naam)
    #geen archiveerbestand

    whitelist = {
        'boerderijen': ['monumentnr', 'complexnr', 'hoofdcategorie', 'subcategorie', 'url', 'mon_omschrijving_500', 'in_welke_set', 'geometry']
    }

    databron_databestand_naam = "Boerderijen_3dec24.gdb"
    databron_databestand_type = "ESRI File Geodatabases"
    databron_databestand_api = "fileshare"   # fileshare of url
    databron_databestand_fileshare = os.path.normpath(r"data//")


### Rijksmonumentale sluizen en stuwen
lokaal bestand/ netwerkschijf

In [295]:
if dataset == "Rijksmonumentale sluizen en stuwen":
    dataset_naam = "Rijksmonumentale sluizen en stuwen"
    database_naam = toDatabaseNamingConventions(dataset_naam)

#geen gecomprimeerd bestand

    whitelist = {
        'sluizen_stuwen': ["monumentnr","complex","legenda","naam","foto_thumb","foto_groot","fotograaf","plaats","versie","rmon_url","geometry"]
    } # let op, complex heeft een alias: complexnr. Maar in de db heet deze "complex".

    databron_databestand_naam = "sluizen_stuwen.gdb"
    databron_databestand_type = "ESRI File Geodatabases"
    databron_databestand_api = "fileshare"   # fileshare of url
    databron_databestand_fileshare = os.path.normpath(r"data//")

### Rijksbeschermde groenaanleggen
lokaal bestand / netwerkschijf

In [296]:
if dataset == "Rijksbeschermde groenaanleggen":
    dataset_naam = "Rijksbeschermde groenaanleggen"
    database_naam = toDatabaseNamingConventions(dataset_naam)

#geen gecomprimeerd bestand

    whitelist = {
        'rijksmonumentaal_groen': ["monumentnr","complex","categorie","cat_legenda","naam","type_aanleg","foto_thumb","foto_groot","fotograaf","adres","postcode","plaats","provincie","contour","bron_contour","versie","Shape_Length","Shape_Area","geometry"]
    }

    databron_databestand_naam = "rijksmonumentaal_groen_13mrt23.gdb"
    databron_databestand_type = "ESRI File Geodatabases"
    databron_databestand_api = "fileshare"   # fileshare of url
    databron_databestand_fileshare = os.path.normpath(r"data//")

### Bestuurlijke Gebieden

#### 2026
GeoPackage vanaf ATOM service

In [297]:
if dataset == "Bestuurlijke Gebieden 2026":
    dataset_naam = "Bestuurlijke Gebieden 2026"
    database_naam = toDatabaseNamingConventions(dataset_naam)

#geen gecomprimeerd bestand

    whitelist = {
            'landgebied': ['identificatie', 'naam', 'code', 'geometry'],
            'provinciegebied': ['identificatie', 'naam', 'code', 'ligt_in_land_code', 'ligt_in_land_naam', 'geometry'],
            'gemeentegebied': ['identificatie', 'naam', 'code', 'ligt_in_provincie_code', 'ligt_in_provincie_naam', 'geometry']
        }

    databron_databestand_naam = "BestuurlijkeGebieden_2026.gpkg"
    databron_databestand_type = "GeoPackage" #GeoPackage, Shapefile, ESRI File Geodatabases, GeoJSON of GML
    databron_databestand_api = "url"   # fileshare of url
    databron_databestand_url = "https://service.pdok.nl/kadaster/brk-bestuurlijke-gebieden/atom/downloads/"
    databron_databestand_atom_feed = "https://service.pdok.nl/kadaster/brk-bestuurlijke-gebieden/atom/bestuurlijke_gebieden.xml"


#### 2025
GeoPackage vanaf ATOM service

In [298]:
if dataset == "Bestuurlijke Gebieden 2025":
    dataset_naam = "Bestuurlijke Gebieden 2025"
    database_naam = toDatabaseNamingConventions(dataset_naam)

#geen gecomprimeerd bestand

    whitelist = {
            'landgebied': ['identificatie', 'naam', 'code', 'geometry'],
            'provinciegebied': ['identificatie', 'naam', 'code', 'ligt_in_land_code', 'ligt_in_land_naam', 'geometry'],
            'gemeentegebied': ['identificatie', 'naam', 'code', 'ligt_in_provincie_code', 'ligt_in_provincie_naam', 'geometry']
        }

    databron_databestand_naam = "BestuurlijkeGebieden_2025.gpkg"
    databron_databestand_type = "GeoPackage" #GeoPackage, Shapefile, ESRI File Geodatabases, GeoJSON of GML
    databron_databestand_api = "url"   # fileshare of url
    databron_databestand_url = "https://service.pdok.nl/kadaster/brk-bestuurlijke-gebieden/atom/downloads/"
    databron_databestand_atom_feed = "https://service.pdok.nl/kadaster/brk-bestuurlijke-gebieden/atom/bestuurlijke_gebieden.xml"

#### 2024
GeoPackage vanaf ATOM service

In [299]:
if dataset == "Bestuurlijke Gebieden 2024":
    dataset_naam = "Bestuurlijke Gebieden 2024"
    database_naam = toDatabaseNamingConventions(dataset_naam)

#geen gecomprimeerd bestand

    whitelist = {
            'landgebied': ['identificatie', 'naam', 'code', 'geometry'],
            'provinciegebied': ['identificatie', 'naam', 'code', 'ligt_in_land_code', 'ligt_in_land_naam', 'geometry'],
            'gemeentegebied': ['identificatie', 'naam', 'code', 'ligt_in_provincie_code', 'ligt_in_provincie_naam', 'geometry']
        }

    databron_databestand_naam = "BestuurlijkeGebieden_2024.gpkg"
    databron_databestand_type = "GeoPackage" #GeoPackage, Shapefile, ESRI File Geodatabases, GeoJSON of GML
    databron_databestand_api = "url"   # fileshare of url
    databron_databestand_url = "https://service.pdok.nl/kadaster/brk-bestuurlijke-gebieden/atom/downloads/"
    databron_databestand_atom_feed = "https://service.pdok.nl/kadaster/brk-bestuurlijke-gebieden/atom/bestuurlijke_gebieden.xml"


#### 2023
GeoPackage vanaf ATOM service

In [300]:
if dataset == "Bestuurlijke Gebieden 2023":
    dataset_naam = "Bestuurlijke Gebieden 2023"
    database_naam = toDatabaseNamingConventions(dataset_naam)

    whitelist = {
            'landgebied': ['identificatie', 'naam', 'code', 'geometry'],
            'provinciegebied': ['identificatie', 'naam', 'code', 'ligt_in_land_code', 'ligt_in_land_naam', 'geometry'],
            'gemeentegebied': ['identificatie', 'naam', 'code', 'ligt_in_provincie_code', 'ligt_in_provincie_naam', 'geometry']
        }

    databron_databestand_naam = "BestuurlijkeGebieden_2023.gpkg"
    databron_databestand_type = "GeoPackage" #GeoPackage, Shapefile, ESRI File Geodatabases, GeoJSON of GML
    databron_databestand_api = "url"   # fileshare of url
    databron_databestand_url = "https://service.pdok.nl/kadaster/brk-bestuurlijke-gebieden/atom/downloads/"
    databron_databestand_atom_feed = "https://service.pdok.nl/kadaster/brk-bestuurlijke-gebieden/atom/bestuurlijke_gebieden.xml"


#### 2022
GeoPackage vanaf ATOM service

In [301]:
if dataset == "Bestuurlijke Gebieden 2022":
    dataset_naam = "Bestuurlijke Gebieden 2022"
    database_naam = toDatabaseNamingConventions(dataset_naam)

    whitelist = {
            'landgebied': ['identificatie', 'naam', 'code', 'geometry'],
            'provinciegebied': ['identificatie', 'naam', 'code', 'ligt_in_land_code', 'ligt_in_land_naam', 'geometry'],
            'gemeentegebied': ['identificatie', 'naam', 'code', 'ligt_in_provincie_code', 'ligt_in_provincie_naam', 'geometry']
        }

    databron_databestand_naam = "BestuurlijkeGebieden_2022.gpkg"
    databron_databestand_type = "GeoPackage" #GeoPackage, Shapefile, ESRI File Geodatabases, GeoJSON of GML
    databron_databestand_api = "url"   # fileshare of url
    databron_databestand_url = "https://service.pdok.nl/kadaster/brk-bestuurlijke-gebieden/atom/downloads/"
    databron_databestand_atom_feed = "https://service.pdok.nl/kadaster/brk-bestuurlijke-gebieden/atom/bestuurlijke_gebieden.xml"

#### 2021
GeoPackage vanaf ATOM service

In [302]:
if dataset == "Bestuurlijke Gebieden 2021":
    dataset_naam = "Bestuurlijke Gebieden 2021"
    database_naam = toDatabaseNamingConventions(dataset_naam)

    whitelist = {
            'landgebied': ['identificatie', 'naam', 'code', 'geometry'],
            'provinciegebied': ['identificatie', 'naam', 'code', 'ligt_in_land_code', 'ligt_in_land_naam', 'geometry'],
            'gemeentegebied': ['identificatie', 'naam', 'code', 'ligt_in_provincie_code', 'ligt_in_provincie_naam', 'geometry']
        }

    databron_databestand_naam = "BestuurlijkeGebieden_2021.gpkg"
    databron_databestand_type = "GeoPackage" #GeoPackage, Shapefile, ESRI File Geodatabases, GeoJSON of GML
    databron_databestand_api = "url"   # fileshare of url
    databron_databestand_url = "https://service.pdok.nl/kadaster/brk-bestuurlijke-gebieden/atom/downloads/"
    databron_databestand_atom_feed = "https://service.pdok.nl/kadaster/brk-bestuurlijke-gebieden/atom/bestuurlijke_gebieden.xml"

#### Bestuurlijke grenzen 2021

NOG NIET GEIMPLEMENTEERD

In [303]:
if dataset == "Bestuurlijke grenzen 2021":
    dataset_naam = "Bestuurlijke grenzen 2021"
    database_naam = toDatabaseNamingConventions(dataset_naam)

    databron_archiveerbestand_naam = "bestuurlijkegrenzen_gml_2021.zip"
    databron_archiveerbestand_type =  "zip"
    databron_archiveerbestand_wachtwoord = ""
    databron_archiveerbestand_api = "url"    # fileshare of url
    databron_archiveerbestand_fileshare = ""
    databron_archiveerbestand_url = "https://service.pdok.nl/kadaster/brk-bestuurlijke-grenzen/atom/downloads/"
    databron_archiveerbestand_atom_feed = "https://service.pdok.nl/kadaster/brk-bestuurlijke-grenzen/atom/bestuurlijke_grenzen.xml"

    whitelist = {
            '': ['', '', '', '', '', '', '', '']
        }

    databron_databestand_naam = ""
    databron_databestand_namen = ["Provinciegrenzen.gml", "Landsgrens.gml", "Gemeentegrenzen.gml"]
    databron_databestand_type = "GML" #GeoPackage, Shapefile, ESRI File Geodatabases, GeoJSON of GML
    databron_databestand_wachtwoord = ""
    databron_databestand_api = ""   # fileshare of url
    databron_databestand_fileshare = ""
    databron_databestand_url = ""
    databron_databestand_atom_feed = ""
    databron_wfs_parameters = ""


#### Bestuurlijke grenzen 2020
NOG NIET GEIMPLEMENTEERD

In [304]:
if dataset == "Bestuurlijke grenzen 2020":
    dataset_naam = "Bestuurlijke grenzen 2020"
    database_naam = toDatabaseNamingConventions(dataset_naam)

    databron_archiveerbestand_naam = "bestuurlijkegrenzen_2020.zip"
    databron_archiveerbestand_type =  "zip"
    databron_archiveerbestand_wachtwoord = ""
    databron_archiveerbestand_api = "url"    # fileshare of url
    databron_archiveerbestand_fileshare = ""
    databron_archiveerbestand_url = "https://service.pdok.nl/kadaster/brk-bestuurlijke-grenzen/atom/downloads/"
    databron_archiveerbestand_atom_feed = "https://service.pdok.nl/kadaster/brk-bestuurlijke-grenzen/atom/bestuurlijke_grenzen.xml"

    whitelist = {
            '': ['', '', '', '', '', '', '', '']
        }

    databron_databestand_naam = ""
    databron_databestand_namen = ["Provinciegrenzen.gml", "Landsgrens.gml", "Gemeentegrenzen.gml"]
    databron_databestand_type = "GML" #GeoPackage, Shapefile, ESRI File Geodatabases, GeoJSON of GML
    databron_databestand_wachtwoord = ""
    databron_databestand_api = ""   # fileshare of url
    databron_databestand_fileshare = ""
    databron_databestand_url = ""
    databron_databestand_atom_feed = ""
    databron_wfs_parameters = ""


#### Bestuurlijke grenzen 2019
NOG NIET GEIMPLEMENTEERD

In [305]:
if dataset == "Bestuurlijke grenzen 2019":
    dataset_naam = "Bestuurlijke grenzen 2019"
    database_naam = toDatabaseNamingConventions(dataset_naam)

    databron_archiveerbestand_naam = "bestuurlijkegrenzen_2019.zip"
    databron_archiveerbestand_type =  "zip"
    databron_archiveerbestand_wachtwoord = ""
    databron_archiveerbestand_api = "url"    # fileshare of url
    databron_archiveerbestand_fileshare = ""
    databron_archiveerbestand_url = "https://service.pdok.nl/kadaster/brk-bestuurlijke-grenzen/atom/downloads/"
    databron_archiveerbestand_atom_feed = "https://service.pdok.nl/kadaster/brk-bestuurlijke-grenzen/atom/bestuurlijke_grenzen.xml"

    whitelist = {
            '': ['', '', '', '', '', '', '', '']
        }

    databron_databestand_naam = ""
    databron_databestand_namen = ["Provinciegrenzen.gml", "Landsgrens.gml", "Gemeentegrenzen.gml"]
    databron_databestand_type = "GML" #GeoPackage, Shapefile, ESRI File Geodatabases, GeoJSON of GML
    databron_databestand_wachtwoord = ""
    databron_databestand_api = ""   # fileshare of url
    databron_databestand_fileshare = ""
    databron_databestand_url = ""
    databron_databestand_atom_feed = ""
    databron_wfs_parameters = ""


#### Bestuurlijke grenzen 2018
NOG NIET GEIMPLEMENTEERD

In [306]:
if dataset == "Bestuurlijke grenzen 2018":
    dataset_naam = "Bestuurlijke grenzen 2018"
    database_naam = toDatabaseNamingConventions(dataset_naam)

    databron_archiveerbestand_naam = "bestuurlijkegrenzen_2018.zip"
    databron_archiveerbestand_type =  "zip"
    databron_archiveerbestand_wachtwoord = ""
    databron_archiveerbestand_api = "url"    # fileshare of url
    databron_archiveerbestand_fileshare = ""
    databron_archiveerbestand_url = "https://service.pdok.nl/kadaster/brk-bestuurlijke-grenzen/atom/downloads/"
    databron_archiveerbestand_atom_feed = "https://service.pdok.nl/kadaster/brk-bestuurlijke-grenzen/atom/bestuurlijke_grenzen.xml"

    whitelist = {
            '': ['', '', '', '', '', '', '', '']
        }

    databron_databestand_naam = ""
    databron_databestand_namen = ["Provinciegrenzen.gml", "Landsgrens.gml", "Gemeentegrenzen.gml"]
    databron_databestand_type = "GML" #GeoPackage, Shapefile, ESRI File Geodatabases, GeoJSON of GML
    databron_databestand_wachtwoord = ""
    databron_databestand_api = ""   # fileshare of url
    databron_databestand_fileshare = ""
    databron_databestand_url = ""
    databron_databestand_atom_feed = ""
    databron_wfs_parameters = ""


#### Bestuurlijke grenzen 2017
NOG NIET GEIMPLEMENTEERD

In [307]:
if dataset == "Bestuurlijke grenzen 2017":
    dataset_naam = "Bestuurlijke grenzen 2017"
    database_naam = toDatabaseNamingConventions(dataset_naam)

    databron_archiveerbestand_naam = "bestuurlijkegrenzen_2017.zip"
    databron_archiveerbestand_type =  "zip"
    databron_archiveerbestand_wachtwoord = ""
    databron_archiveerbestand_api = "url"    # fileshare of url
    databron_archiveerbestand_fileshare = ""
    databron_archiveerbestand_url = "https://service.pdok.nl/kadaster/brk-bestuurlijke-grenzen/atom/downloads/"
    databron_archiveerbestand_atom_feed = "https://service.pdok.nl/kadaster/brk-bestuurlijke-grenzen/atom/bestuurlijke_grenzen.xml"

    whitelist = {
            '': ['', '', '', '', '', '', '', '']
        }

    databron_databestand_naam = ""
    databron_databestand_namen = ["Provinciegrenzen.gml", "Landsgrens.gml", "Gemeentegrenzen.gml"]
    databron_databestand_type = "GML" #GeoPackage, Shapefile, ESRI File Geodatabases, GeoJSON of GML
    databron_databestand_wachtwoord = ""
    databron_databestand_api = ""   # fileshare of url
    databron_databestand_fileshare = ""
    databron_databestand_url = ""
    databron_databestand_atom_feed = ""
    databron_wfs_parameters = ""


### Basisregistratie Adressen en Gebouwen (BAG)
GeoPackage van ATOM service (LET OP! Groot bestand!)

In [308]:
if dataset == "Basisregistratie Adressen en Gebouwen":
    dataset_naam = "Basisregistratie Adressen en Gebouwen"
    database_naam = toDatabaseNamingConventions(dataset_naam)

    whitelist = {
            'woonplaats': ['rdf_seealso', 'identificatie', 'status', 'woonplaats', 'bronhouder_identificatie', 'geometry'],
            'verblijfsobject': ['rdf_seealso', 'identificatie', 'oppervlakte', 'status', 'gebruiksdoel', 'openbare_ruimte_naam', 'openbare_ruimte_naam_kort', 'huisnummer', 'huisletter', 'toevoeging', 'postcode', 'woonplaats_naam', 'bouwjaar', 'pand_identificatie', 'pandstatus', 'nummeraanduiding_hoofdadres_identificatie', 'openbare_ruimte_identificatie', 'woonplaats_identificatie', 'bronhouder_identificatie', 'geometry'],
            'pand': ['rdf_seealso', 'rdf_seealso', 'identificatie', 'bouwjaar', 'status', 'gebruiksdoel', 'oppervlakte_min', 'oppervlakte_max', 'aantal_verblijfsobjecten', 'geometry']
        }

    databron_databestand_naam = "bag-light.gpkg"
    databron_databestand_type = "GeoPackage" #GeoPackage, Shapefile, ESRI File Geodatabases, GeoJSON of GML
    databron_databestand_api = "url"   # fileshare of url
    databron_databestand_url = "https://service.pdok.nl/kadaster/bag/atom/downloads/"
    databron_databestand_atom_feed = "https://service.pdok.nl/kadaster/bag/atom/bag.xml"

### Publiekrechtelijke Beperkingen (Wkpb)
GeoPackage van ATOM service

In [309]:
if dataset == "Publiekrechtelijke Beperkingen":
    dataset_naam = "Publiekrechtelijke Beperkingen"
    database_naam = toDatabaseNamingConventions(dataset_naam)

    whitelist = {
        'pb_multipolygon': ['identificatie', 'domein', 'code_groep', 'naam_groep', 'grondslag_code', 'grondslag_omschrijving', 'type_beperkingsgebied', 'datum_in_werking', 'datum_beeindiging', 'datum_aanbieding', 'tijdstip_inschrijven_kadaster', 'gebaseerd_op_stuk', 'datum_kenbaarheid', 'meerdere_brondocumenten_ingeschreven', 'statutaire_naam_rechtspersoon', 'monumentnummer', 'organisatie_informatie_nummer_oin', 'geometry'],
        'pb_multilinestring': ['identificatie', 'domein', 'code_groep', 'naam_groep', 'grondslag_code', 'grondslag_omschrijving', 'type_beperkingsgebied', 'datum_in_werking', 'datum_beeindiging', 'datum_aanbieding', 'tijdstip_inschrijven_kadaster', 'gebaseerd_op_stuk', 'datum_kenbaarheid', 'meerdere_brondocumenten_ingeschreven', 'statutaire_naam_rechtspersoon', 'monumentnummer', 'organisatie_informatie_nummer_oin', 'geometry']
        }

    databron_databestand_naam = "Publiekrechtelijkebeperkingen.gpkg"
    databron_databestand_type = "GeoPackage" #GeoPackage, Shapefile, ESRI File Geodatabases, GeoJSON of GML
    databron_databestand_api = "url"   # fileshare of url
    databron_databestand_url = "https://service.pdok.nl/kadaster/brk-publiekrechtelijke-beperkingen-wkpb/atom/downloads/"
    databron_databestand_atom_feed = "https://service.pdok.nl/kadaster/brk-publiekrechtelijke-beperkingen-wkpb/atom/publiekrechtelijke_beperkingen_wkpb.xml"

### Rijksmonumentenregister
lokaal gecomprimeerde MS Access database

In [310]:
if dataset == "Rijksmonumentenregister":
    dataset_naam = "Rijksmonumentenregister"
    database_naam = toDatabaseNamingConventions(dataset_naam)

    databron_archiveerbestand_naam = "Extract_MRS.zip"
    databron_archiveerbestand_type =  ".zip"
    databron_archiveerbestand_api = "fileshare"    # fileshare of url
    databron_archiveerbestand_fileshare = os.path.join(r"data//")

    whitelist = {
        'tblTEXT_OBJECT' : ['TXO_TEXT_KEY', 'OBJ_NUMMER',  'TXO_CREATIE_DATUM', 'TXO_IND_ACT_HIST', 'KIO_CREATIE_DATUM',
                            'TXO_ACT_DATUM', 'TXO_SOORT','TXO_TEKST', 'TXO_SOORT_ZKP']
    } # TODO whitelist nog niet compleet

    databron_databestand_naam = "Extract_MRS_V11.0.05.mdb"
    databron_databestand_type = "MS access database" #GeoPackage, Shapefile, ESRI File Geodatabases, GeoJSON of GML
    databron_databestand_api = "fileshare"   # fileshare of url

### Rijksmonumentencontouren
alias RCE dicos

GML van WFS service

In [311]:
if dataset == "rijksmonumentencontouren":
    dataset_naam = "rijksmonumentencontouren"
    database_naam = toDatabaseNamingConventions(dataset_naam)

    databron_archiveerbestand_naam = ""
    databron_archiveerbestand_type =  ""
    databron_archiveerbestand_wachtwoord = ""
    databron_archiveerbestand_api = ""    # fileshare of url
    databron_archiveerbestand_fileshare = ""
    databron_archiveerbestand_url = ""
    databron_archiveerbestand_atom_feed = ""

    # whitelist = {}
    whitelist = {}

    databron_databestand_naam = "rijksmonumentencontouren"
    databron_databestand_namen = []
    databron_databestand_type = "GML" #GeoPackage, Shapefile, ESRI File Geodatabases, GeoJSON of GML
    databron_databestand_wachtwoord = ""
    databron_databestand_api = "url"   # fileshare of url
    databron_databestand_fileshare = ""
    databron_databestand_url = "https://data.geo.cultureelerfgoed.nl/openbaar/wfs?"
    databron_databestand_atom_feed = ""
    databron_wfs_parameters = dict(
        service="WFS",
        version="2.0.0",
        request="GetFeature",
        typeName="openbaar:rijksmonumentcontouren",
        outputFormat="application/gml+xml; version=3.2",
    )
    databron_ngr_metadata_code = "eedd39d6-b427-4e11-90fd-f5054241c3a7"



### Rijksmonumentenpunten
GML van WFS service

In [312]:
if dataset == "rijksmonumentenpunten":
    dataset_naam = "rijksmonumentenpunten"
    database_naam = toDatabaseNamingConventions(dataset_naam)

    databron_archiveerbestand_naam = ""
    databron_archiveerbestand_type =  ""
    databron_archiveerbestand_wachtwoord = ""
    databron_archiveerbestand_api = ""    # fileshare of urlbobo
    databron_archiveerbestand_fileshare = ""
    databron_archiveerbestand_url = ""
    databron_archiveerbestand_atom_feed = ""

    # whitelist = {}
    whitelist = {}

    databron_databestand_naam = "rijksmonumentenpunten"
    databron_databestand_namen = []
    databron_databestand_type = "GML" #GeoPackage, Shapefile, ESRI File Geodatabases, GeoJSON of GML
    databron_databestand_wachtwoord = ""
    databron_databestand_api = "url"   # fileshare of url
    databron_databestand_fileshare = ""
    databron_databestand_url = "https://data.geo.cultureelerfgoed.nl/openbaar/wfs?"
    databron_databestand_atom_feed = ""
    databron_wfs_parameters = dict(
        service="WFS",
        version="2.0.0",
        request="GetFeature",
        typeName="openbaar:rijksmonumentpunten",
        outputFormat="application/gml+xml; version=3.2",
    )

### Archeologische Complexen (DEF_complexen_v8)

In [313]:
if dataset == "archeologische complexen": # naam die overeenkomt met 1 uit de lijst met namen uit vorige code chuck
    dataset_naam = "DEF_complexen_v8" # naam waaronder de dataset bekend staat (doorgaans hetzelfde als dataset)
    database_naam = toDatabaseNamingConventions(dataset_naam) # schoon de naam op

# geen zip bestand

# De whitelist bevat een dictionary met daarin alle tabellen, en alle kolommen binnen die tabellen, die moeten worden gedownload. Bij GIS bestanden is er altijd een kolom "geometry"
    whitelist = {}
    # de "databestand" variabelen moeten worden ingevuld voor het in te laden bestand. Als het gaat om een gearchiveerd bestand, gaat het om het bestand binnen de gecomprimeerde map dat de data ook werkelijk bevat.
    databron_databestand_naam = "DEF_complexen_v8.shp" # file naam, inclusief uitgang, zonder path
    databron_databestand_type = "Shapefile" #type file, kan nu zijn: GeoPackage, Shapefile, ESRI File Geodatabases, GeoJSON, GML of MS access database
    databron_databestand_api = "fileshare"   # fileshare of url
    databron_databestand_namen = ["DEF_complexen_v8.cpg", "DEF_complexen_v8.dbf", "DEF_complexen_v8.prj", "DEF_complexen_v8.sbn", "DEF_complexen_v8.sbx", "DEF_complexen_v8.shx"] # als er meerdere bestanden gecombineerd gedownload/geontzipt worden, geef hier de lijst van bestanden.
    databron_databestand_fileshare = os.path.normpath(r".\data\DEF_complexen_v8") # zie hierboven, maar dan voor bestand. ZONDER bestandsnaam!

# Timestamp van het brondata vaststellen
Wanneer is het bestand voor het laatst bewerkt? Dit is belangrijk om bij te houden voor versiebeheer en data time travel.

In [314]:
# het path naar het archiveerbestand of databestand in de databron wordt samengesteld
# vanaf deze locatie wordt de het bestand gedownload


if databron_databestand_atom_feed != "" or databron_archiveerbestand_atom_feed != "":
    # haal de timestamp een databestand of een archiveerbestand uit een atom feed
    databron_timestamp = get_timestamp_from_atom_feed(databron_archiveerbestand_atom_feed, databron_databestand_atom_feed, databron_archiveerbestand_naam, databron_databestand_naam)
elif databron_archiveerbestand_api == "fileshare":
    # de datum waarop het archiveerbestand gewijzigd is, wordt vastgesteld
    databron_archiveerbestand_path = os.path.join(databron_archiveerbestand_fileshare,databron_archiveerbestand_naam)
    databron_timestamp = modification_date(databron_archiveerbestand_path)
elif databron_databestand_api == "fileshare":
    databron_databestand_path = os.path.join(databron_databestand_fileshare,databron_databestand_naam)
    # de datum waarop het databestand gewijzigd is, wordt vastgesteld
    databron_timestamp = modification_date(databron_databestand_path)
elif databron_wfs_parameters != "":
    # voor WFS moet de databron uit de bijgeleverde metadata komen, deze wordt los via de API van het nationaal georegister opgehaald. Als de metadata geen modification date heeft, probeer het dan op te halen uit de WFS

    # TODO does NOT work. Somehow the API keeps returning html instead of xml. temporarily return today's date
    # databron_timestamp = get_timestamp_from_ngr_rest_api(databron_ngr_metadata_code)
    # print(databron_timestamp)
    # if databron_timestamp == "daily":
    #     databron_timestamp = datetime.date.today()
    databron_timestamp = datetime.date.today()
else:
    pass
    
print("databron_timestamp", databron_timestamp)
print("databron_archiveerbestand_path", databron_archiveerbestand_path)
print("databron_databestand_path", databron_databestand_path)
print("databron_archiveerbestand_atom_feed", databron_archiveerbestand_atom_feed)
print("databron_databestand_atom_feed", databron_databestand_atom_feed)
print("databron_archiveerbestand_naam", databron_archiveerbestand_naam)
print("databron_databestand_naam", databron_databestand_naam)
print("databron_wfs_parameters", databron_wfs_parameters)



1785230484.6138332
databron_timestamp 2026-07-28 11:21:24.613833
databron_archiveerbestand_path 
databron_databestand_path data\DEF_complexen_v8\DEF_complexen_v8.shp
databron_archiveerbestand_atom_feed 
databron_databestand_atom_feed 
databron_archiveerbestand_naam 
databron_databestand_naam DEF_complexen_v8.shp
databron_wfs_parameters 


# Download de brondata wanneer deze nog niet in het datalake aanwezig is

In [315]:
# het jaar, de maand en de dag in de vorm van een string worden uit de opslag datum gehaald
jaar = databron_timestamp.strftime("%Y")
maand = databron_timestamp.strftime("%m")
dag = databron_timestamp.strftime("%d")
datalake_archiveerbestand_path = ""
datalake_databestand_path = ""
print(jaar, maand, dag)



if databron_archiveerbestand_path != "":
    datalake_archiveerbestand_path = os.path.normpath(f"{datalake_path}/{ruwe_laag_path}/{database_naam}/{jaar}/{maand}/{dag}/{databron_archiveerbestand_naam}")
    if os.path.isfile(datalake_archiveerbestand_path):
        print(f"{datalake_archiveerbestand_path} exists")
    else:
        copy_file(databron_archiveerbestand_path, datalake_archiveerbestand_path, databron_archiveerbestand_type)
        print(f"{databron_archiveerbestand_path} is copied to {datalake_archiveerbestand_path}")
elif databron_databestand_path != "":
    datalake_databestand_path = os.path.normpath(f"{datalake_path}/{ruwe_laag_path}/{database_naam}/{jaar}/{maand}/{dag}/{databron_databestand_naam}")
    if os.path.isfile(datalake_databestand_path) or os.path.isdir(datalake_databestand_path):
        print(f"{datalake_databestand_path} exists")
    else:
        copy_file(databron_databestand_path, datalake_databestand_path, databron_databestand_type)
        print(f"{databron_databestand_path} is copied to {datalake_databestand_path}")
elif databron_archiveerbestand_url != "":
    databron_archiveerbestand_download = os.path.join(databron_archiveerbestand_url,databron_archiveerbestand_naam)
    print(databron_archiveerbestand_download)
    datalake_archiveerbestand_path = os.path.normpath(f"{datalake_path}/{ruwe_laag_path}/{database_naam}/{jaar}/{maand}/{dag}/{databron_archiveerbestand_naam}")
    print(datalake_archiveerbestand_path)
    if os.path.isfile(datalake_archiveerbestand_path):
        print(f"{datalake_archiveerbestand_path} exists")
    else:
        # download bestand en sla het bestand op
        req = requests.get(databron_archiveerbestand_download)
        req.raise_for_status()
        # get the directory out of the path
        dir_name, _ = os.path.split(datalake_archiveerbestand_path)
        # create directory if does not exist 
        os.makedirs(dir_name,exist_ok=True)
        with open(datalake_archiveerbestand_path, 'wb') as f:
            f.write(req.content)
        print(f"{databron_archiveerbestand_download} is copied to {datalake_archiveerbestand_path}")
elif databron_wfs_parameters != "":
    # download bestand (alle features!) van WFS
    datalake_databestand_path = os.path.normpath(f"{datalake_path}/{ruwe_laag_path}/{database_naam}/{jaar}/{maand}/{dag}/{databron_databestand_naam}.gml")
    print(datalake_databestand_path)
    if os.path.isfile(datalake_databestand_path) or os.path.isdir(datalake_databestand_path):
        print(f"{datalake_databestand_path} exists")
    else:
        dir_name, _ = os.path.split(datalake_databestand_path)
        os.makedirs(dir_name,exist_ok=True)
        req = requests.get(databron_databestand_url, params=databron_wfs_parameters)
        req.raise_for_status()
        with open(datalake_databestand_path, "wb") as f:
            f.write(req.content)
            print(f"download van WFS {dataset_naam} naar {datalake_databestand_path}")

elif databron_databestand_url != "":
    databron_databestand_download = os.path.join(databron_databestand_url,databron_databestand_naam)
    print(databron_databestand_download)
    datalake_databestand_path = os.path.normpath(f"{datalake_path}/{ruwe_laag_path}/{database_naam}/{jaar}/{maand}/{dag}/{databron_databestand_naam}")
    print(datalake_databestand_path)
    if os.path.isfile(datalake_databestand_path) or os.path.isdir(datalake_databestand_path):
        print(f"{datalake_databestand_path} exists")
    else:
        # download bestand en sla het bestand op
        req = requests.get(databron_databestand_download)
        req.raise_for_status()
        # get the directory out of the path
        dir_name, _ = os.path.split(datalake_databestand_path)
        # create directory if does not exist 
        os.makedirs(dir_name,exist_ok=True)
        with open(datalake_databestand_path, 'wb') as f:
            f.write(req.content)
        print(f"{databron_databestand_download} is copied to {datalake_databestand_path}")
else:
    pass



2026 07 28
data\DEF_complexen_v8\DEF_complexen_v8.shp data\ruw\def_complexen_v8\2026\07\28\DEF_complexen_v8.shp
data\DEF_complexen_v8\DEF_complexen_v8.shp
data\ruw\def_complexen_v8\2026\07\28\DEF_complexen_v8.shp
data\DEF_complexen_v8\DEF_complexen_v8.cpg
data\ruw\def_complexen_v8\2026\07\28\DEF_complexen_v8.cpg
data\DEF_complexen_v8\DEF_complexen_v8.dbf
data\ruw\def_complexen_v8\2026\07\28\DEF_complexen_v8.dbf
data\DEF_complexen_v8\DEF_complexen_v8.prj
data\ruw\def_complexen_v8\2026\07\28\DEF_complexen_v8.prj
data\DEF_complexen_v8\DEF_complexen_v8.sbn
data\ruw\def_complexen_v8\2026\07\28\DEF_complexen_v8.sbn
data\DEF_complexen_v8\DEF_complexen_v8.sbx
data\ruw\def_complexen_v8\2026\07\28\DEF_complexen_v8.sbx
data\DEF_complexen_v8\DEF_complexen_v8.shx
data\ruw\def_complexen_v8\2026\07\28\DEF_complexen_v8.shx
data\DEF_complexen_v8\DEF_complexen_v8.shp is copied to data\ruw\def_complexen_v8\2026\07\28\DEF_complexen_v8.shp


# Convert layers in database to Parquet files

In [316]:

if datalake_archiveerbestand_path != "":
    # het is een zip bestand
    if zip_has_pwd(datalake_archiveerbestand_path) and datalake_archiveerbestand_path != "":
        # zip file is password protected
        # create temporary zip file without password protection
        temp_file_path = remove_pwd_from_zip(datalake_archiveerbestand_path, databron_archiveerbestand_wachtwoord)
        # create parquet file  from every layer in a temporary zip file without password protection
        #print(temp_file_path)
        if databron_databestand_naam.endswith(".mdb") or databron_databestand_naam.endswith(".accdb"):
            print("file is a zipped MS access file with password")
            table_list = access_to_parquet(temp_file_path, datalake_databestand_path, whitelist)
            print("table list", table_list)
            # het is een gezipt accessbestand.
        else:
            gis_file_to_parquet(temp_file_path, datalake_databestand_path, whitelist)
        # remove temporary zip file without password protection
        os.remove(temp_file_path)
    else:
        # zip file is not password protected
        # create parquet file from every layer in a zip file without password protection
        #print(datalake_archiveerbestand_path, datalake_databestand_path)
        if databron_databestand_naam.endswith(".mdb") or databron_databestand_naam.endswith(".accdb"):
            print("File is a zipped MS access file")
            table_list = access_to_parquet(datalake_archiveerbestand_path, databron_databestand_naam, whitelist)
            print("table list", table_list)
            # het is een gezipt accessbestand.
        else:
            gis_file_to_parquet(datalake_archiveerbestand_path, databron_databestand_naam, whitelist)
else: 
    # het is geen zip bestand
    # print("a", datalake_archiveerbestand_path)
    # print("b", datalake_databestand_path)
    # print("c", whitelist)

    if databron_databestand_naam.endswith(".mdb") or databron_databestand_naam.endswith(".accdb"):
        # TODO Non-zip bestand nog niet getest! (zou in principe niet nodig moeten zijn bij automatisch downloaden vanaf RCE server, dan krijg je sowieso een ZIP bestand. Dus eventjes gelaten voor wat het is.
        print("file is a zipped MS access file")
        table_list = access_to_parquet(datalake_archiveerbestand_path, datalake_databestand_path, whitelist)
        print(table_list)
    else:
        gis_file_to_parquet(datalake_archiveerbestand_path, datalake_databestand_path, whitelist)

layer DEF_complexen_v8
